# Structured Pruning of Gemma-3 270M

We compare MLP channel, attention head, and decoder block pruning using Ukrainian validation loss.

In [1]:
import torch
import torch_pruning as tp

from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import dotenv
from pathlib import Path

import pandas as pd
from rouge_score import rouge_scorer
import re
import unicodedata

dotenv.load_dotenv()

True

## Load XL-SUM dataset for validation

We need to validate pruned model to get the best score. We do not need all the dataset, there is a need to take at least 100 samples.

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

validation_dataset = load_dataset(
    "parquet",
    data_files={
        "validation": str(PROJECT_ROOT / "data" / "raw" / "xlsum" / "ukrainian" / "validation" / "0000.parquet")
    },
    split="validation",
)

In [3]:
print(validation_dataset[0])

{'id': 'news-41032282', 'url': 'https://www.bbc.com/ukrainian/news-41032282', 'title': 'Саудівська поліція відпустила підлітка, який танцював "макарену"', 'summary': 'Саудівська поліція відпустила 14-річного підлітка, якого затримали після публікації в інтернеті \u200eвідео, на якому юнак танцює "макарену" на дорозі. Йому не висунули жодних звинувачень.\u200e', 'text': 'Міністерство внутрішніх справ країни заявило, що юнак підписав письмову обіцянку того, що більше "не буде залучений до поведінки, яка може поставити під загрозу його життя та життя інших".\u200e У поліції не назвали його ім\'я, проте зазначили, що він є громадянином іншої арабської країни.\u200e \u200e45-секундний ролик був знятий минулого року. На ньому юнак перешкоджає рухові автомобілів, \u200eтанцюючи в навушниках на п\'ятисмуговій дорозі у саудівському місті Джидда.\u200e Юнака затримали у цей вівторок у місті Мекка - після того, як ролик раптово набув шаленої \u200eпопулярності у соцмережах. Його допитали з привод

In [4]:
validation_subset = validation_dataset.shuffle(seed=42).select(range(100))
val_data = validation_subset["text"]

rouge_subset = validation_subset.select(range(20))
rouge_data = rouge_subset["text"]
rouge_summaries = rouge_subset["summary"]

## Load Gemma-3 270M

In [5]:
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)

print("Model loaded on:", model.device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
W1024 20:07:13.514000 90639 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cpu


In [6]:
def inference(model, prompt):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7
        )

    return inputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

The function for running validation set

In [7]:
def evaluate_model(model, tokenizer):
    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    total_tokens = 0
    for prompt in val_data:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                labels=inputs["input_ids"],
                use_cache=False,
            )

        token_count = inputs["input_ids"].shape[1] - 1
        total_loss += outputs.loss.item() * token_count
        total_tokens += token_count

    return { "loss": total_loss / total_tokens }

In [8]:
baseline_loss = evaluate_model(model, tokenizer)
baseline_loss

{'loss': 3.762071979278721}

We need to compute how many storage it takes to run the model

In [9]:
def model_size_mb(model):
    return model.get_memory_footprint() / 1_000_000

In [10]:
print(f"{model_size_mb(model)} MB model size")

1072.394756 MB model size


We also need an additional metric to evaluate performance of pruned models. The best metric for XL-SUM dataset is Rouge. It represents the ability of the model to summarize the text (comparison between LLM summary and human summary).

In [11]:
class UnicodeWordTokenizer:
    def tokenize(self, text):
        text = unicodedata.normalize("NFC", text).lower()
        text = text.replace("’", "'").replace("ʼ", "'")
        return re.findall(r"[^\W_]+(?:'[^\W_]+)*", text)


rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    tokenizer=UnicodeWordTokenizer(),
)


def xlsum_prompt_formation(text):
    text = text.replace("\u200e", "")
    text = re.sub(r" {2,}", " ", text)
    return f"Підсумуй цю статтю:\n{text}\n"


def generate_summary(model, text):
    model.eval()
    prompt = xlsum_prompt_formation(text)
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def evaluate_rouge(model):
    model.eval()
    rouge_results = []

    for text, reference in zip(rouge_data, rouge_summaries):
        prediction = generate_summary(model, text)
        scores = rouge.score(target=reference, prediction=prediction)
        rouge_results.append({
            name: score.fmeasure
            for name, score in scores.items()
        })

    details = pd.DataFrame(rouge_results)
    return details[["rouge1", "rouge2", "rougeL"]].mean().to_dict()


def model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_model_size, pruned_model, method_name):
    loss = evaluate_model(pruned_model, tokenizer)["loss"]
    rouge_scores = evaluate_rouge(pruned_model)

    print(f"\nBaseline model size (MB): {baseline_model_size:.3f}")
    print(f"{method_name} model size (MB): {model_size_mb(pruned_model):.3f}")
    print(f"Baseline loss: {baseline_model_loss:.4f}")
    print(f"{method_name} loss: {loss:.4f}\n")
    print(
        "Baseline Rouge: "
        f"Rouge1: {baseline_model_rouge['rouge1']:.4f}, "
        f"Rouge2: {baseline_model_rouge['rouge2']:.4f}, "
        f"RougeL: {baseline_model_rouge['rougeL']:.4f}"
    )
    print(
        f"{method_name} Rouge: "
        f"Rouge1: {rouge_scores['rouge1']:.4f}, "
        f"Rouge2: {rouge_scores['rouge2']:.4f}, "
        f"RougeL: {rouge_scores['rougeL']:.4f}"
    )
    return {"loss": loss, "size_mb": model_size_mb(pruned_model), **rouge_scores}

### Baseline ROUGE

ROUGE compares generated summaries with the XL-Sum reference summaries. Higher values are better.

In [12]:
baseline_loss = baseline_loss["loss"]
baseline_rouge = evaluate_rouge(model)
baseline_size = model_size_mb(model)

pd.DataFrame([baseline_rouge], index=["Baseline"]).round(4)

,rouge1,rouge2,rougeL
Baseline,0.1037,0.0268,0.0824


In [13]:
def describe_model(model):
    block = model.model.layers[0]
    print("Model:", type(model).__name__)
    print("MLP width:", block.mlp.gate_proj.out_features)
    print("Query width:", block.self_attn.q_proj.out_features)
    print("Blocks:", len(model.model.layers))

In [14]:
tokenized_prompt, answer = inference(model, prompt="Answer in one word what is 1 + 1")
print(answer)

Answer in one word what is 1 + 1 = 2?

**Answer:** 2



## Model structure

Initial state of the model

In [15]:
print(model.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [16]:
describe_model(model)

Model: Gemma3ForCausalLM
MLP width: 2048
Query width: 1024
Blocks: 18


In [17]:
print(f"Parameters before pruning: {tp.utils.count_params(model)}")

Parameters before pruning: 268098176


## 1. Smallest MLP channels

**Goal:** Remove 10% of MLP channels.

### Method
Rank channels by L2 norm and remove the smallest ones jointly from gate, up, and down projections.

In [18]:
from copy import deepcopy
from torch import nn


def forward_fn(model, input_ids):
    return model(input_ids=input_ids, use_cache=False, return_dict=False)[0]


def prune_channels(model, ratio=0.25):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [19]:
pruned_model = prune_channels(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.17.pre_feedforward_layernorm.weight', 'model.layers.14.input_layernorm.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.post_feedforward_layernorm.weight', 'model.layers.5.self_attn.q_norm.weight', 'model.layers.8.post_attention_layernorm.weight', 'model.layers.11.post_attention_layernorm.weight', 'model.layers.12.post_feedforward_layernorm.weight', 'model.layers.16.self_attn.k_norm.weight', 'model.layers.3.self_attn.q_norm.weight', 'model.layers.0.post_feedforward_layernorm.weight', 'model.layers.9.self_attn.k_norm.weight', 'model.layers.17.post_attention_layernorm.weight', 'model.layers.16.input_layernorm.weight', 'model.layers.15.pre_feedforward_layernorm.weight', 'model.layers.10.self_attn.q_norm.weight', 'model.layers.0.self_attn.q_norm.we

In [20]:
print(inference(pruned_model, "Hello, how are you?")[1])

Hello, how are you? ***ign

 AS
 on
,
 as a constant,
,
,
      in a limited and normales b as global on, for
on  and jny,  so, we,
   Wyn'ass and
unpat on the 2  
  ,  ,   3  
,1
   
  un, and,

un-,                


In [21]:
describe_model(pruned_model)

Model: Gemma3ForCausalLM
MLP width: 1840
Query width: 1024
Blocks: 18


In [22]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model)}")

Parameters after pruning: 260909696


In [23]:
print(f"{model_size_mb(pruned_model)} MB model size")

1043.640836 MB model size


### Results

In [24]:
print("Original:", baseline_loss)
print("Pruned:  ", evaluate_model(pruned_model, tokenizer))

Original: 3.762071979278721
Pruned:   {'loss': 22.69308587813449}


In [25]:
rouge_smallest = evaluate_rouge(pruned_model)
pd.DataFrame([rouge_smallest], index=["Smallest MLP"]).round(4)

,rouge1,rouge2,rougeL
Smallest MLP,0.0014,0.0,0.0014


### Conclusion
260.91M parameters; 1043.64 MB; Loss 22.693

Smallest-norm channels are a poor choice. Loss increased about 6×.

## 2. Largest MLP channels

**Goal:** Remove 10% of MLP channels using the opposite ranking.

### Method
Rank channels by L2 norm and remove the largest ones jointly from gate, up, and down projections.

In [26]:
def prune_model_2(model, ratio=0.1):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2, normalizer=lambda s: -s),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [27]:
pruned_model2 = prune_model_2(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.12.post_feedforward_layernorm.weight', 'model.layers.16.input_layernorm.weight', 'model.layers.6.self_attn.k_norm.weight', 'model.layers.0.post_feedforward_layernorm.weight', 'model.layers.3.self_attn.q_norm.weight', 'model.layers.4.pre_feedforward_layernorm.weight', 'model.layers.11.input_layernorm.weight', 'model.layers.7.post_attention_layernorm.weight', 'model.layers.14.self_attn.q_norm.weight', 'model.layers.17.input_layernorm.weight', 'model.layers.8.post_feedforward_layernorm.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.4.self_attn.q_norm.weight', 'model.layers.11.post_attention_layernorm.weight', 'model.layers.14.pre_feedforward_layernorm.weight', 'model.layers.12.post_attention_layernorm.weight', 'model.layers.6.post_attention_layernorm.weight', 'model.layers.10.self_attn.q_norm.weight', 'mo

In [28]:
print(inference(pruned_model2, "Hello, how are you?")[1])

Hello, how are you????????????????????????????؟?؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟؟


In [29]:
describe_model(pruned_model2)

Model: Gemma3ForCausalLM
MLP width: 1840
Query width: 1024
Blocks: 18


In [30]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model2)}")

Parameters after pruning: 260909696


In [31]:
print(f"{model_size_mb(pruned_model2)} MB model size")

1043.640836 MB model size


### Results

In [32]:
print("Original:", baseline_loss)
print("Pruned:  ", evaluate_model(pruned_model2, tokenizer))

Original: 3.762071979278721
Pruned:   {'loss': 9.054247155563816}


In [33]:
rouge_largest = evaluate_rouge(pruned_model2)
pd.DataFrame([rouge_largest], index=["Largest MLP"]).round(4)

,rouge1,rouge2,rougeL
Largest MLP,0.0022,0.0,0.0011


### Conclusion
260.91M parameters; 1043.64 MB; Loss 9.054

Better than the first pruning method, but loss is still about 3× worse.

## 3. Random MLP channels

**Goal:** Remove 10% of MLP channels without magnitude ranking.

### Method
Select channels randomly and remove each one jointly from gate, up, and down projections.

In [34]:
def prune_model3(model, ratio=0.1, seed=42):
    model = deepcopy(model)

    torch.manual_seed(seed)

    for block in model.model.layers:
        n_prune = int(block.mlp.gate_proj.out_features * ratio)
        idx = torch.randperm(block.mlp.gate_proj.out_features)[:n_prune].tolist()

        tp.prune_linear_out_channels(block.mlp.gate_proj, idx)
        tp.prune_linear_out_channels(block.mlp.up_proj, idx)
        tp.prune_linear_in_channels(block.mlp.down_proj, idx)

    return model

In [35]:
pruned_model3 = prune_model3(model)

In [36]:
print(inference(pruned_model3, "Hello, how are you?")[1])

Hello, how are you?
Is there a chance of rain tomorrow evening?
I'm trying to find out what's going on with the news about the new ringgit (Silver Crystal Entacson).
I've just learned about it, but I'm not sure about the details.
What are your thoughts on what will happen if the ringgit falls?
I'm planning on to go out tonight, but I'm not sure if I'll see what happens.
I


In [37]:
describe_model(pruned_model3)

Model: Gemma3ForCausalLM
MLP width: 1844
Query width: 1024
Blocks: 18


### Results

In [38]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model3)}")

Parameters after pruning: 261047936


In [39]:
print("Original:", baseline_loss)
print("Pruned:  ", evaluate_model(pruned_model3, tokenizer))

Original: 3.762071979278721
Pruned:   {'loss': 6.439094986779418}


In [40]:
print(f"{model_size_mb(pruned_model3)} MB model size")

1044.193796 MB model size


In [41]:
rouge_random = evaluate_rouge(pruned_model3)
pd.DataFrame([rouge_random], index=["Random MLP"]).round(4)

,rouge1,rouge2,rougeL
Random MLP,0.0018,0.0,0.0018


### Conclusion
261.05M parameters; 1044.19 MB; Loss 6.439

This is the best MLP cut, although loss still increased by almost 2×.

## 4. Attention heads

**Goal:** Find the least important query head.

### Method
Remove each whole query head from `q_proj` and the matching inputs of `o_proj`, then compare validation loss.

In [42]:
def prune_heads(model, heads_num):
    model = deepcopy(model)

    head_size = model.config.head_dim
    remaining_heads = model.config.num_attention_heads - len(heads_num)

    delete_range = [i for head in heads_num for i in range(head * head_size, (head + 1) * head_size)]

    for block in model.model.layers:
        attn = block.self_attn
        q_proj, o_proj = attn.q_proj, attn.o_proj
        tp.prune_linear_out_channels(q_proj, delete_range)
        tp.prune_linear_in_channels(o_proj, delete_range)
        attn.num_key_value_groups = remaining_heads

    model.config.num_attention_heads = remaining_heads
    return model

In [43]:
pruned_model4 = prune_heads(model, [0])

In [44]:
describe_model(pruned_model4)

Model: Gemma3ForCausalLM
MLP width: 2048
Query width: 768
Blocks: 18


### Results

In [45]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model4)}")

Parameters after pruning: 262199936


In [46]:
print("Original:", baseline_loss)

for head in range(model.config.num_attention_heads):
    new_model = prune_heads(model, [head])
    print(f"Pruned (deleted head: {head}):  ", evaluate_model(new_model, tokenizer))

Original: 3.762071979278721
Pruned (deleted head: 0):   {'loss': 6.9507034381028}
Pruned (deleted head: 1):   {'loss': 10.358075373957778}
Pruned (deleted head: 2):   {'loss': 7.416869957818614}
Pruned (deleted head: 3):   {'loss': 4.964996977196919}


In [47]:
print(f"{model_size_mb(pruned_model4)} MB model size")

1048.801796 MB model size


### Heads 0 and 2 together

In [48]:
pruned_model_head02 = prune_heads(model, [0, 2])
print("Pruned (deleted head: [0, 2]):  ", evaluate_model(pruned_model_head02, tokenizer))

Pruned (deleted head: [0, 2]):   {'loss': 13.903848714339134}


In [49]:
rouge_head3 = evaluate_rouge(prune_heads(model, [3]))
rouge_heads_02 = evaluate_rouge(pruned_model_head02)
pd.DataFrame(
    [rouge_head3, rouge_heads_02],
    index=["Head 3", "Heads 0 and 2"],
).round(4)

,rouge1,rouge2,rougeL
Head 3,0.0192,0.0017,0.0192
Heads 0 and 2,0.0055,0.0000,0.0055


### Conclusion
262.20M parameters; 1048.80 MB; Loss 4.965

Head 3 causes the smallest individual loss increase. Removing two heads hurts much more than removing one.

## 5. Whole blocks

**Goal:** Demonstrate structured pruning of one complete decoder block.

### Method
Remove block 0 and inspect the resulting architecture, parameter count, and model size.

In [50]:
def prune_blocks(model, blocks_num):
    model = deepcopy(model)
    drop = set(blocks_num)

    model.model.layers = nn.ModuleList(block for i, block in enumerate(model.model.layers) if i not in drop)
    model.config.layer_types = [kind for i, kind in enumerate(model.config.layer_types) if i not in drop]
    model.config.num_hidden_layers = len(model.model.layers)

    for new_index, block in enumerate(model.model.layers):
        block.layer_idx = new_index
        block.self_attn.layer_idx = new_index

    return model

### Results

In [51]:
pruned_model5 = prune_blocks(model, [0])

In [52]:
describe_model(pruned_model5)

Model: Gemma3ForCausalLM
MLP width: 2048
Query width: 1024
Blocks: 17


In [53]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model5)}")

Parameters after pruning: 262524544


In [54]:
print(f"{model_size_mb(pruned_model5)} MB model size")

1050.100228 MB model size


In [55]:
rouge_block0 = evaluate_rouge(pruned_model5)
pd.DataFrame([rouge_block0], index=["Block 0"]).round(4)

,rouge1,rouge2,rougeL
Block 0,0.0,0.0,0.0


### Conclusion
262.52M parameters; 1050.10 MB; Loss 13.741

Removing a complete block reduces model depth without changing the width of the remaining layers.

## 6. Several blocks

**Goal:** Identify low-impact decoder blocks and remove several of them together.

### Method
First remove every block separately and compare validation loss. Then remove the best candidates—blocks 6, 7, 8, and 9—together.

### Results

In [56]:
print("Original:", baseline_loss)

for block in range(len(model.model.layers)):
    new_model = prune_blocks(model, [block])
    print(f"Pruned (deleted block: {block}):  ", evaluate_model(new_model, tokenizer))

Original: 3.762071979278721
Pruned (deleted block: 0):   {'loss': 13.741168926346347}
Pruned (deleted block: 1):   {'loss': 24.491463463272286}
Pruned (deleted block: 2):   {'loss': 5.552441718188603}
Pruned (deleted block: 3):   {'loss': 13.217021415697515}
Pruned (deleted block: 4):   {'loss': 4.632076993436385}
Pruned (deleted block: 5):   {'loss': 14.033922541834055}
Pruned (deleted block: 6):   {'loss': 4.236766526314408}
Pruned (deleted block: 7):   {'loss': 4.092196664731873}
Pruned (deleted block: 8):   {'loss': 4.1274701213601475}
Pruned (deleted block: 9):   {'loss': 3.956135412754908}
Pruned (deleted block: 10):   {'loss': 5.131008808513201}
Pruned (deleted block: 11):   {'loss': 5.817351503255666}
Pruned (deleted block: 12):   {'loss': 5.795061940326086}
Pruned (deleted block: 13):   {'loss': 6.285704697240472}
Pruned (deleted block: 14):   {'loss': 5.244687169081186}
Pruned (deleted block: 15):   {'loss': 5.065710371413165}
Pruned (deleted block: 16):   {'loss': 4.67000282

In [57]:
pruned_model_blocks = prune_blocks(model, [6, 7, 8, 9])
print("Pruned (deleted block: [6, 7, 8, 9]):  ", evaluate_model(pruned_model_blocks, tokenizer))
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model_blocks)}")

Pruned (deleted block: [6, 7, 8, 9]):   {'loss': 4.8277655851846655}
Parameters after pruning: 245803648


In [58]:
print(f"{model_size_mb(pruned_model_blocks)} MB model size")

983.216644 MB model size


In [59]:
rouge_blocks_6789 = evaluate_rouge(pruned_model_blocks)
pd.DataFrame([rouge_blocks_6789], index=["Blocks 6-9"]).round(4)

,rouge1,rouge2,rougeL
Blocks 6-9,0.0226,0.0,0.0226


### Conclusion
Blocks 6–9; 245.80M parameters; 983.22 MB; Loss 4.828

This is the best overall quality–size trade-off.

## 7. MLP channels + Attention heads + Blocks

**Goal:** Test whether the three pruning methods work well together.

### Method
Apply block pruning first, then attention-head pruning, and finally random MLP-channel pruning.

#### Step 1: Block pruning

In [60]:
fully_pruned_model = deepcopy(model)

fully_pruned_model = prune_blocks(fully_pruned_model, [7, 9])

print(f"Parameters after block pruning: {tp.utils.count_params(fully_pruned_model)}")
print("Pruned (deleted block: [7, 9]):  ", evaluate_model(fully_pruned_model, tokenizer))

Parameters after block pruning: 256950912
Pruned (deleted block: [7, 9]):   {'loss': 4.710071361609793}


#### Step 2: Attention-head pruning

In [61]:
for head in range(fully_pruned_model.config.num_attention_heads):
    new_model = prune_heads(fully_pruned_model, [head])
    print(f"Pruned (deleted head: {head}):  ", evaluate_model(new_model, tokenizer))

Pruned (deleted head: 0):   {'loss': 6.837079068702576}
Pruned (deleted head: 1):   {'loss': 9.888875634721773}
Pruned (deleted head: 2):   {'loss': 12.116002185079504}
Pruned (deleted head: 3):   {'loss': 7.249733160246522}


Head 0 has the lowest loss after block pruning, so it is selected for the next step.

In [62]:
fully_pruned_model = prune_heads(fully_pruned_model, [0])

print(f"Parameters after block and head pruning: {tp.utils.count_params(fully_pruned_model)}")
print("Pruned (deleted head: [0]):  ", evaluate_model(fully_pruned_model, tokenizer))

Parameters after block and head pruning: 251708032
Pruned (deleted head: [0]):   {'loss': 6.837079068702576}


#### Step 3: MLP-channel pruning

In [63]:
fully_pruned_model2 = prune_model3(fully_pruned_model, ratio=0.1)

print(f"Parameters after block and head and MLP channel pruning: {tp.utils.count_params(fully_pruned_model2)}")
print("Pruned:  ", evaluate_model(fully_pruned_model2, tokenizer))

Parameters after block and head and MLP channel pruning: 245441152
Pruned:   {'loss': 9.473133885940287}


### Overall Results
245.44M parameters; 981.77 MB; Loss 9.473

In [64]:
print(f"{model_size_mb(fully_pruned_model2)} MB model size")

981.76666 MB model size


In [65]:
rouge_combined = evaluate_rouge(fully_pruned_model2)
pd.DataFrame([rouge_combined], index=["Combined"]).round(4)

,rouge1,rouge2,rougeL
Combined,0.0214,0.0,0.02


### Conclusion
Combining all three methods is worse than removing four low-impact blocks alone.

## Results

ROUGE for every stage. Each score is the mean F1 on the same 20 XL-Sum summaries.


In [66]:
rouge_table = pd.DataFrame(
    [
        baseline_rouge,
        rouge_smallest,
        rouge_largest,
        rouge_random,
        rouge_head3,
        rouge_heads_02,
        rouge_block0,
        rouge_blocks_6789,
        rouge_combined,
    ],
    index=[
        "Baseline",
        "Smallest MLP",
        "Largest MLP",
        "Random MLP",
        "Head 3",
        "Heads 0 and 2",
        "Block 0",
        "Blocks 6-9",
        "Combined",
    ],
)
rouge_table.round(4)

,rouge1,rouge2,rougeL
Baseline,0.1037,0.0268,0.0824
Smallest MLP,0.0014,0.0000,0.0014
Largest MLP,0.0022,0.0000,0.0011
Random MLP,0.0018,0.0000,0.0018
Head 3,0.0192,0.0017,0.0192
Heads 0 and 2,0.0055,0.0000,0.0055
Block 0,0.0000,0.0000,0.0000
Blocks 6-9,0.0226,0.0000,0.0226
Combined,0.0214,0.0000,0.0200


Even though the numbers are not so high, the ROUGE agrees with the loss ranking. Blocks 6-9 keep the most summary overlap (RougeL 0.0226 versus 0.0824), and the combined model is slightly worse at 0.02.

| Experiment | Best configuration | Parameters | Model size (MB) | Validation loss |
|---|---|---:|---:|---:|
| Baseline | Original model | 268.10M | 1072.39 | 3.762 |
| MLP channels | Random 10% | 261.05M | 1044.19 | 6.439 |
| Attention heads | Head 3 | 262.20M | 1048.80 | 4.965 |
| Single block | Block 9 | 262.52M | 1050.10 | 3.956 |
| Several blocks | Blocks 6, 7, 8, and 9 | 245.80M | 983.22 | 4.828 |
| Combined methods | Blocks 7 and 9 + head 0 + random 10% MLP | 245.44M | 981.77 | 9.473 |

Result: Removing blocks 6–9 gives the best overall trade-off: about 22.3M fewer parameters with loss increasing from 3.762 to 4.828. Combining block, head, and MLP pruning makes the model only slightly smaller, while loss rises to 9.473.

## Download the best model

The best model -- model from the 6-th experiment

In [ ]:
MODEL_PATH = PROJECT_ROOT / "models" / "best_pruned_model"

pruned_model_blocks.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)